In [1]:
%pip install keras
%pip install tensorflow

In [2]:


import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle


In [3]:
import numpy as np

data_path = '.'
fname = 'timewise_5s_500p'

X = np.load(f'{data_path}/{fname}_X.npy')  # shape: (N, T)
y = np.load(f'{data_path}/{fname}_y.npy')  # shape: (N,)

X_reserve_test = np.load(f'{data_path}/{fname}_reserve_X.npy')
y_reserve_test = np.load(f'{data_path}/{fname}_reserve_y.npy')

print(X.shape, y.shape)

shuffle(X, y, random_state=123)

print(np.bincount(y))


(23237, 500, 3) (23237,)
[  0 286 192 224 308 215 174 199 353 182 427  89 196 205 454 291 360 193
 331 198 408 402 392 289 399 262 287 331 280 110 314 346 191 292 290 181
 184 213 280 308 204 226 176 267 199 123 283  87  94 334 123 457 407 385
 187 202 351 168 307 171 268 183 360 166 191 190 175 374 281 239  87 202
  89   0 214 277 328 172 278 297 261 329 406 196 174 173 279 317 293 195
 265 214  77 300]


In [4]:
print(X_reserve_test.shape, y_reserve_test.shape)


print(np.bincount(y_reserve_test))

(2851, 500, 3) (2851,)
[  0  97  95   0   0 105   0  98   0  94   0  87   0  91   0  95   0   0
 112 190   0   0   0  93   0  77   0   0   0   0 100   0  85   0   0   0
   0   0 113   0   0   0  93   0   0   0  97   0   0 109   0   0   0   0
 184   0   0   0   0   0   0   0   0   0  93  95   0   0   0   0   0 202
   0   0   0   0   0   0  90 103  81   0   0  95   0   0   0   0   0   0
   0   0  77]


In [12]:
from sklearn.preprocessing import LabelEncoder
import joblib
# Fit on the full set of labels (before train/test split)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

y_encoded_reserve = label_encoder.transform(y_reserve_test)

joblib.dump(label_encoder, 'label_encoder.pkl')

# Then split the encoded labels
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [13]:
# X shape: (N, T, 3)
X_train_scaled = np.zeros_like(X_train)
X_val_scaled = np.zeros_like(X_val)
X_test_scaled = np.zeros_like(X_test)
X_reserve_scaled = np.zeros_like(X_reserve_test)

scalers = []

for i in range(X_train.shape[2]):  # loop over channels: 0=x, 1=y, 2=z
    scaler = StandardScaler()
    scaler.fit(X_train[:, :, i])  # fit on (N, T) for this channel

    X_train_scaled[:, :, i] = scaler.transform(X_train[:, :, i])
    X_val_scaled[:, :, i]   = scaler.transform(X_val[:, :, i])
    X_test_scaled[:, :, i]  = scaler.transform(X_test[:, :, i])
    X_reserve_scaled[:, :, i]  = scaler.transform(X_reserve_test[:, :, i])

    scalers.append(scaler)

# Optionally save all 3 scalers
joblib.dump(scalers, 'scalers.pkl')


['scalers.pkl']

In [14]:
# Now you can get num_classes safely
num_classes = len(np.unique(y_encoded))

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)
y_reserve_cat = to_categorical(y_encoded_reserve, num_classes)

In [15]:
from sklearn.utils import class_weight
import numpy as np

# Compute weights for each class
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dict for Keras
class_weights_dict = dict(enumerate(class_weights_array))


In [16]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, BatchNormalization, GlobalAveragePooling1D

input_shape = (X_train_scaled.shape[1], X_train_scaled.shape[2])  # (time_steps, features)

inputs = Input(shape=input_shape)

# 🔍 Feature extractor
x = Conv1D(64, kernel_size=5, activation='relu', padding='same')(inputs)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)
x = Dropout(0.2)(x)

x = Conv1D(128, kernel_size=5, activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)
x = Dropout(0.2)(x)

# 🔁 Sequence modeler
x = Bidirectional(LSTM(64, return_sequences=False))(x)
x = Dropout(0.3)(x)

# 🧠 Classifier head
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])



In [17]:
print(X_train_scaled.shape)  # should be (samples, time_steps, features)
print(np.std(X_train_scaled))  # should not be zero or near-zero


(16265, 500, 3)
0.9999999999999993


In [18]:
print(y_train_cat.shape)  # should be (samples, num_classes)
print(np.unique(np.argmax(y_train_cat, axis=1)))  # Should cover multiple labels


(16265, 92)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91]


In [19]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_cnn_lstm.keras", save_best_only=True, monitor="val_loss", mode="min", verbose=1)


history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=32,
    callbacks = [early_stop, checkpoint],
    class_weight=class_weights_dict
)


Epoch 1/100
509/509 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0439 - loss: 4.2767
Epoch 1: val_loss improved from inf to 3.26148, saving model to best_cnn_lstm.keras
509/509 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.0440 - loss: 4.2761 - val_accuracy: 0.2025 - val_loss: 3.2615
Epoch 2/100
508/509 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1842 - loss: 3.1123
Epoch 2: val_loss improved from 3.26148 to 2.50134, saving model to best_cnn_lstm.keras
509/509 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.1843 - loss: 3.1117 - val_accuracy: 0.3267 - val_loss: 2.5013
Epoch 3/100
508/509 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2850 - loss: 2.5196
Epoch 3: val_loss improved from 2.50134 to 1.88310, saving model to best_cnn_lstm.keras
509/509 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.2851 - loss: 2.5191 - val_accuracy: 0.4816 - val_loss: 1.8831
Epoch 4/100
509/509 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3938 - loss: 2.0279
Epoch 4: val_loss improved from 

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dropout, Dense, Input

model2 = Sequential([
    Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])),

    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    LSTM(64),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])


In [ ]:
print(X_test_scaled.shape)

(3914, 500, 1)


In [20]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=1)
print(f"✅ Test accuracy: {test_acc:.2%}")

109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9808 - loss: 0.1009
✅ Test accuracy: 97.88%


In [21]:
test_2_loss, test_2_acc = model.evaluate(X_reserve_scaled, y_reserve_cat, verbose=1)
print(f"Test Reserve accuracy: {test_2_acc:.2%}")

90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2911 - loss: 12.9649
Test Reserve accuracy: 18.52%
